In [ ]:
from arch.bootstrap import MCS
import pandas as pd
import numpy as np
import re
from pathlib import Path

def should_exclude_model_spec(model_label: str, spec_label: str) -> bool:
    """
    Exclude:
      - all IV_SJ versions
      - all RV_SJ versions
      - EGARCH_IV_RV
    """
    if spec_label in {"IV_SJ", "RV_SJ", "IV_CQ", "RV_CQ"}:
        return True

    if model_label == "EGARCH" and spec_label == "IV_RV":
        return True

    return False


# FZ and AL loss functions
def fz_loss(returns: np.ndarray, VaR: np.ndarray, ES: np.ndarray, quantile: float):
    L = (returns < VaR).astype(int)
    term1 = -L * (VaR - returns) / (quantile * ES)
    term2 = VaR / ES
    term3 = np.log(-ES)
    return term1 + term2 + term3 - 1

def al_loss(returns: np.ndarray, VaR: np.ndarray, ES: np.ndarray, quantile: float):
    L = (returns < VaR).astype(int)
    term1 = -np.log((quantile - 1) / ES)
    term2 = -(returns - VaR) * (quantile - L) / (quantile * ES)
    return term1 + term2


# CONFIG
PROJECT_ROOT = Path.cwd().parents[1]
results_dir = PROJECT_ROOT / "src" / "predictions"

alphas = [0.01, 0.025, 0.05, 0.95, 0.975, 0.99]
loss_data = {}

print("Reading predictions from:")
print(results_dir)


# helpers
def pick_first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def resolve_quantile_col(df: pd.DataFrame, prefix: str, alpha: float) -> str | None:
    cands = [
        f"{prefix}_{alpha:.3f}",
        f"{prefix}_{alpha:.2f}",
        f"{prefix}_{alpha:.3f}".rstrip("0").rstrip("."),
        f"{prefix}_{alpha}".rstrip("0").rstrip("."),
    ]
    return pick_first_existing_col(df, cands)

def get_date_col(df: pd.DataFrame) -> str | None:
    return pick_first_existing_col(df, ["Date", "date", "DATE"])

def get_returns_col(df: pd.DataFrame) -> str | None:
    return pick_first_existing_col(df, ["TrueY", "LogReturn", "log_return", "return", "returns"])

def parse_model_and_spec_from_filename(fname: str):
    """
    Handles files like:
      catboost_IV.csv
      catboost_IV_RV.csv
      DB_IV_CJ.csv
      egarch_RV.csv
      lgbm_IV_SJ.csv
      lstm_IV.csv
      QR_RV_SV.csv
      xgb_IV_CQ.csv

    Returns (model_label, spec_label) or None if not recognized.
    Final MCS column name becomes: f"{model_label}_{spec_label}"
    """
    stem = Path(fname).stem
    parts = stem.split("_")

    if len(parts) < 2:
        return None

    model_raw = parts[0].lower()
    spec = "_".join(parts[1:])

    model_map = {
        "db": "DB",
        "qr": "QR",
        "egarch": "EGARCH",
        "catboost": "CatBoost",
        #"xgb": "XGB",
        "lgbm": "LGBM",
        "lstm": "LSTM",
    }

    if model_raw not in model_map:
        return None

    return model_map[model_raw], spec

# MAIN: compute losses for all files
csv_files = sorted(results_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {results_dir.resolve()}")

print(f"Found {len(csv_files)} csv files in {results_dir.resolve()}")

for fpath in csv_files:
    parsed = parse_model_and_spec_from_filename(fpath.name)
    if parsed is None:
        print(f"[SKIP] {fpath.name}: filename pattern not recognized")
        continue

    model_label, spec_label = parsed

    if should_exclude_model_spec(model_label, spec_label):
        print(f"[SKIP] {fpath.name}: excluded by spec filter")
        continue

    col_name = f"{model_label}_{spec_label}"
    print(f"[READ] {fpath.name} -> {col_name}")

    df = pd.read_csv(fpath)

    date_col = get_date_col(df)
    ret_col = get_returns_col(df)

    if date_col is None:
        print(f"[SKIP] {fpath.name}: missing Date/date column")
        continue
    if ret_col is None:
        print(f"[SKIP] {fpath.name}: missing returns column (TrueY/LogReturn/log_return)")
        continue

    dates = pd.to_datetime(df[date_col], errors="coerce")
    returns = pd.to_numeric(df[ret_col], errors="coerce").to_numpy()

    for alpha in alphas:
        q_col = resolve_quantile_col(df, "Quantile", alpha)
        es_col = resolve_quantile_col(df, "ES", alpha)

        if q_col is None or es_col is None:
            continue

        VaR = pd.to_numeric(df[q_col], errors="coerce").to_numpy()
        ES = pd.to_numeric(df[es_col], errors="coerce").to_numpy()
        r = returns.copy()

        alpha_adj = alpha
        right_tail = False

        if alpha > 0.5:
            r = -r
            VaR = -VaR
            ES = -ES
            alpha_adj = 1 - alpha
            right_tail = True

        fz_vals = fz_loss(r, VaR, ES, quantile=alpha_adj)
        al_vals = al_loss(r, VaR, ES, quantile=alpha_adj)

        fz_vals = np.asarray(fz_vals).reshape(-1)
        al_vals = np.asarray(al_vals).reshape(-1)
        mask = (~dates.isna()) & np.isfinite(fz_vals) & np.isfinite(al_vals)

        fz_vals = fz_vals[mask]
        al_vals = al_vals[mask]
        dates_idx = dates[mask]

        suffix = int(alpha_adj * 100)

        if not right_tail:
            names = {f"FZ0_{suffix}": fz_vals, f"AL_{suffix}": al_vals}
        else:
            names = {f"FZ0R_{suffix}": fz_vals, f"ALR_{suffix}": al_vals}

        for loss_name, values in names.items():
            df_new = pd.DataFrame({col_name: values}, index=dates_idx)
            df_new = df_new[~df_new.index.duplicated(keep="first")]

            if loss_name not in loss_data:
                loss_data[loss_name] = df_new
            else:
                loss_data[loss_name] = loss_data[loss_name].join(df_new, how="outer")

print("Finished computing both left & right tail FZ/AL losses.")


# MCS PROCEDURE
mcs_alpha_levels = [0.05, 0.10, 0.25]
loss_fns = list(loss_data.keys())
mcs_results = {}

for alpha in mcs_alpha_levels:
    results_df = None

    for metric in loss_fns:
        df_losses = loss_data[metric].copy().dropna(how="any", axis=0)
        df_losses = df_losses.loc[:, df_losses.nunique() > 1]

        if df_losses.shape[1] < 2:
            print(f"Skipping {metric}: not enough models.")
            continue

        models_valid = df_losses.columns.tolist()
        if results_df is None:
            results_df = pd.DataFrame(index=models_valid, columns=loss_fns)

        loss_matrix = df_losses.to_numpy()
        T = loss_matrix.shape[0]
        block_len = max(2, int(np.sqrt(T)))

        mcs = MCS(
            loss_matrix,
            size=alpha,
            reps=500,
            block_size=block_len,
            bootstrap="stationary",
            method="max",
        )
        mcs.compute()

        included = [df_losses.columns[i] for i in mcs.included]

        for m in df_losses.columns:
            results_df.loc[m, metric] = (m in included)

    mcs_results[alpha] = results_df

# PRINT SUMMARY
LEFT_ORDER = ["FZ0_1", "AL_1", "FZ0_2", "AL_2", "FZ0_5", "AL_5"]
RIGHT_ORDER = ["FZ0R_1", "ALR_1", "FZ0R_2", "ALR_2", "FZ0R_5", "ALR_5"]

LEFT_LABEL = {
    "FZ0_1": ("FZ", "1%"),
    "AL_1":  ("AL", "1%"),
    "FZ0_2": ("FZ", "2.5%"),
    "AL_2":  ("AL", "2.5%"),
    "FZ0_5": ("FZ", "5%"),
    "AL_5":  ("AL", "5%"),
}
RIGHT_LABEL = {
    "FZ0R_1": ("FZ", "99%"),
    "ALR_1":  ("AL", "99%"),
    "FZ0R_2": ("FZ", "97.5%"),
    "ALR_2":  ("AL", "97.5%"),
    "FZ0R_5": ("FZ", "95%"),
    "ALR_5":  ("AL", "95%"),
}

COL_WIDTH = 22

for alpha_mcs, df in mcs_results.items():
    if df is None:
        continue

    conf = 1 - alpha_mcs

    print(f"\n=== MCS SUMMARY (Left Tail, {conf:.0%} confidence) ===")
    left_cols = [c for c in LEFT_ORDER if c in df.columns]

    header = ["Model"] + [f"{LEFT_LABEL[c][0]} ({LEFT_LABEL[c][1]})" for c in left_cols]
    print("".join(h.ljust(COL_WIDTH) for h in header))
    print("-" * (len(header) * COL_WIDTH))

    for model_name in df.index:
        row = [model_name]
        for c in left_cols:
            row.append(str(bool(df.loc[model_name, c])))
        print("".join(r.ljust(COL_WIDTH) for r in row))

    print(f"\n=== MCS SUMMARY (Right Tail, {conf:.0%} confidence) ===")
    right_cols = [c for c in RIGHT_ORDER if c in df.columns]

    header = ["Model"] + [f"{RIGHT_LABEL[c][0]} ({RIGHT_LABEL[c][1]})" for c in right_cols]
    print("".join(h.ljust(COL_WIDTH) for h in header))
    print("-" * (len(header) * COL_WIDTH))

    for model_name in df.index:
        row = [model_name]
        for c in right_cols:
            row.append(str(bool(df.loc[model_name, c])))
        print("".join(r.ljust(COL_WIDTH) for r in row))

# SAVE
# ===============================================================
output_dir = PROJECT_ROOT / "synne" / "mcs_results"
output_dir.mkdir(parents=True, exist_ok=True)

for alpha_mcs, df in mcs_results.items():
    if df is None:
        continue

    conf = 1 - alpha_mcs
    if conf in (0.75, 0.90, 0.95):
        conf_int = int(conf * 100)
        out_path = output_dir / f"MCS_{conf_int}_results_LSTM.csv"
        df.to_csv(out_path)
        print(f"\nSaved {conf_int}% MCS results -> {out_path}\n")

Reading predictions from:
c:\Users\synnerbo\Code\tio4900_master_thesis\src\predictions
Found 14 csv files in C:\Users\synnerbo\Code\tio4900_master_thesis\src\predictions
[READ] LSTM_IV.csv -> LSTM_IV
[READ] LSTM_IV_BF.csv -> LSTM_IV_BF
[READ] LSTM_IV_BF_RR.csv -> LSTM_IV_BF_RR
[READ] LSTM_IV_CJ.csv -> LSTM_IV_CJ
[READ] LSTM_IV_CURVE.csv -> LSTM_IV_CURVE
[READ] LSTM_IV_RR.csv -> LSTM_IV_RR
[READ] LSTM_IV_RV.csv -> LSTM_IV_RV
[READ] LSTM_IV_SLOPE.csv -> LSTM_IV_SLOPE
[READ] LSTM_IV_SLOPE_CURVE.csv -> LSTM_IV_SLOPE_CURVE
[READ] LSTM_IV_SV.csv -> LSTM_IV_SV
[READ] LSTM_RV.csv -> LSTM_RV
[READ] LSTM_RV_CJ.csv -> LSTM_RV_CJ
[READ] LSTM_RV_SV.csv -> LSTM_RV_SV
[SKIP] xgb_predictions_RV_EURUSD_2000ws_IVVol.csv: filename pattern not recognized
Finished computing both left & right tail FZ/AL losses.

=== MCS SUMMARY (Left Tail, 95% confidence) ===
Model                 FZ (1%)               AL (1%)               FZ (2.5%)             AL (2.5%)             FZ (5%)               AL (5%)           